# Project 1 — AI vs. Human Text Detection

Course: Intro to AI & Agents, Summer I 2026  
Labels used in the data: `0 = Human-written text`, `1 = AI-written text`.

This notebook is organized using the four sections required for the project:

1. Data Exploration & Preprocessing
2. Feature Engineering
3. Model Training & Tuning
4. Evaluation & Comparison

The Streamlit app is in `app.py`, and the training code is in `train_models.py`.


## 1. Data Exploration & Preprocessing

I started by loading the Excel dataset, keeping only the `text` and `label` columns, cleaning blank rows, and checking the class balance. The saved model run uses a balanced sample of 2,500 rows so the training can finish on a normal laptop.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('..').resolve()
sys.path.append(str(PROJECT_ROOT))

from project_utils import clean_text, tokenize, compute_linguistic_features, linguistic_feature_names

DATA_PATH = PROJECT_ROOT / 'data' / 'training_data' / 'train_data_with_labels.xlsx'
df = pd.read_excel(DATA_PATH)
df = df[['text', 'label']].copy()
df['text'] = df['text'].apply(clean_text)
df = df[df['text'].str.len() > 0].copy()
df['label'] = df['label'].astype(int)

print(df.shape)
df['label'].value_counts().sort_index()


In [ ]:
balance = df['label'].value_counts().sort_index()
balance.plot(kind='bar', title='Class Balance')
plt.xticks([0, 1], ['Human (0)', 'AI (1)'], rotation=0)
plt.ylabel('Number of samples')
plt.show()

df['word_count'] = df['text'].apply(lambda x: len(tokenize(x)))
df.groupby('label')['word_count'].describe()


### Preprocessing notes

The cleaning step is simple on purpose. I normalize whitespace and fix some encoding issues, but I keep punctuation because punctuation and sentence structure can be useful writing-style signals.


In [ ]:
sample_text = df.loc[0, 'text']
print(sample_text[:500])
print(tokenize(sample_text)[:30])


## 2. Feature Engineering

I compared three feature types:

- **TF-IDF:** word and phrase frequency features.
- **Word2Vec average embeddings:** word vectors trained on the dataset.
- **Linguistic features:** word count, sentence count, sentence length, vocabulary richness, punctuation, uppercase ratio, digit ratio, and readability.

The traditional ML models use TF-IDF plus linguistic features because that gave the best balance between accuracy and explanation.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from scipy.sparse import csr_matrix, hstack

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['text'].values,
    df['label'].values,
    test_size=0.2,
    stratify=df['label'].values,
    random_state=42,
)

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    stop_words='english',
    sublinear_tf=True,
)
X_train_tfidf = vectorizer.fit_transform(X_train_text)
X_test_tfidf = vectorizer.transform(X_test_text)

scaler = StandardScaler()
X_train_ling = scaler.fit_transform(compute_linguistic_features(X_train_text))
X_test_ling = scaler.transform(compute_linguistic_features(X_test_text))

X_train_combined = hstack([X_train_tfidf, csr_matrix(X_train_ling)], format='csr')
X_test_combined = hstack([X_test_tfidf, csr_matrix(X_test_ling)], format='csr')

print(X_train_combined.shape)


### Feature comparison results

These results are saved from the training script in `reports/feature_comparison.csv`.

| feature_set                 |   accuracy |   precision |   recall |    f1 |   roc_auc |
|:----------------------------|-----------:|------------:|---------:|------:|----------:|
| TF-IDF                      |      0.93  |       0.909 |    0.956 | 0.932 |     0.983 |
| Linguistic Features         |      0.722 |       0.696 |    0.788 | 0.739 |     0.772 |
| Word2Vec Average Embeddings |      0.61  |       0.608 |    0.62  | 0.614 |     0.646 |


In [ ]:
feature_results = pd.read_csv(PROJECT_ROOT / 'reports' / 'feature_comparison.csv')
feature_results


## 3. Model Training & Tuning

The project requires six classifiers, so I trained three traditional machine learning models and three deep learning models.

Traditional ML models:
- SVM
- Decision Tree
- AdaBoost

Deep learning models:
- FNN
- LSTM
- CNN for text

For the traditional ML models, I used `GridSearchCV` with F1-score as the tuning metric. For the deep learning models, I used a small manual grid because FNN, LSTM, and CNN take longer to train than the scikit-learn models on a regular laptop. The deep learning tuning checks sequence length, embedding size, hidden units or filters, dropout, learning rate, and epochs. The selected configurations are saved in the final model files used by the Streamlit app.


In [ ]:
# Run this from the notebook if you want to retrain everything.
# It may take a few minutes depending on the computer.

# !python ../train_models.py

# Faster testing run:
# !python ../train_models.py --fast


In [ ]:
model_results = pd.read_csv(PROJECT_ROOT / 'reports' / 'model_comparison.csv')
tuning_plan = pd.read_csv(PROJECT_ROOT / 'reports' / 'tuning_plan.csv')
deep_tuning = pd.read_csv(PROJECT_ROOT / 'reports' / 'deep_tuning_results.csv')

model_results[['model', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'best_params']]


In [ ]:
print('Traditional ML and deep learning tuning plan:')
display(tuning_plan)

print('Selected deep learning configurations used in the saved app models:')
display(deep_tuning)


### Tuning notes

For the scikit-learn models, `GridSearchCV` tests the parameter grid and keeps the setting with the best F1-score. I used F1-score because the project is a classification task and I wanted a balanced metric between precision and recall.

For the deep learning models, I did not leave the models at default settings. I tested small, realistic configurations instead of a very large search. This was more practical for this project because LSTM and CNN models can take longer to train. The tuning plan is saved in `reports/tuning_plan.csv`, and the selected deep learning settings are saved in `reports/deep_tuning_results.csv`.

The final saved models are the ones loaded by `app.py`.


## 4. Evaluation & Comparison

I evaluated every model with accuracy, precision, recall, F1-score, confusion matrix, and ROC-AUC. I focused mostly on F1-score because it gives a balanced view of precision and recall.


### Model comparison results

| model         |   accuracy |   precision |   recall |    f1 |   roc_auc |
|:--------------|-----------:|------------:|---------:|------:|----------:|
| SVM           |      0.938 |       0.936 |    0.94  | 0.938 |     0.989 |
| AdaBoost      |      0.914 |       0.888 |    0.948 | 0.917 |     0.976 |
| FNN           |      0.876 |       0.87  |    0.884 | 0.877 |     0.952 |
| CNN           |      0.866 |       0.865 |    0.868 | 0.866 |     0.942 |
| Decision Tree |      0.844 |       0.812 |    0.896 | 0.852 |     0.836 |
| LSTM          |      0.756 |       0.75  |    0.768 | 0.759 |     0.792 |


In [ ]:
plot_df = model_results.sort_values('f1', ascending=True)
plt.figure(figsize=(8, 4))
plt.barh(plot_df['model'], plot_df['f1'])
plt.xlabel('F1-score')
plt.title('Model F1-score Comparison')
plt.xlim(0, 1)
plt.show()


### ROC curve comparison

![ROC model comparison](../reports/figures/roc_model_comparison.png)


### Confusion matrices

SVM:

![SVM confusion matrix](../reports/figures/confusion_svm.png)

Decision Tree:

![Decision Tree confusion matrix](../reports/figures/confusion_decision_tree.png)

AdaBoost:

![AdaBoost confusion matrix](../reports/figures/confusion_adaboost.png)

FNN:

![FNN confusion matrix](../reports/figures/confusion_fnn.png)

LSTM:

![LSTM confusion matrix](../reports/figures/confusion_lstm.png)

CNN:

![CNN confusion matrix](../reports/figures/confusion_cnn.png)


### Written analysis

The best model in my saved run was **SVM** with an F1-score of **0.938**. SVM performed strongly because TF-IDF creates a high-dimensional text representation, and SVM usually works well with that type of data.

The linguistic features by themselves were weaker than TF-IDF, but they still helped explain the text style. Features like sentence length, vocabulary richness, punctuation ratio, and readability gave extra signals that are easier to understand than word weights alone.

The deep learning models worked, but they were more sensitive to training time and sequence length. The CNN and FNN performed better than the LSTM in this saved run. My main takeaway is that simpler ML models can be very strong for this type of text classification, especially when the dataset is not huge.

I would not use this kind of tool as the only decision-maker in a classroom. It can be useful as a warning signal, but AI detection is not perfect. A human review would still be needed before making any serious academic decision.


For tuning, the traditional ML models used GridSearchCV. The deep learning models used a smaller tuning grid because they take longer to train, and the final selected configurations are documented in the reports folder. This made the project more realistic to run on my own computer while still comparing different hyperparameter choices.
